In [ ]:
import pandas as pd
import geopandas
import contextily as cx
import requests
import httpx
import asyncio
import shelve
from geopy.geocoders import GoogleV3

In [163]:
key_file = 'keys/googlemaps_key.txt'
with open(key_file) as key:
    API_KEY = key.read().strip()

In [13]:
geolocator = GoogleV3(api_key=API_KEY)

In [14]:
geo_components = {
    'country':'us',
    'administrative_area':'Allegheny County, Pennsylvania'
}

In [334]:
SEM = asyncio.Semaphore(20)

### Address Point Dupes

In [ ]:
address_points = geopandas.GeoDataFrame(pd.read_csv('data/processing/AddressPoints_dupes.csv', index_col=0))
address_points = address_points.drop(columns=['geo_x', 'geo_y'])
address_points.head(1)

,ADDRESS_ID,PARENT_ID,STREET_ID,ADDRESS_TY,STATUS,ADDR_NUM_P,ADDR_NUM,ADDR_NUM_S,ST_PREMODI,ST_PREFIX,...,SOURCE,EXP_FLAG,FULL_ADDRE,ST_SUFFIX,POINT_X,POINT_Y,geometry,COMBINED_ADDR,FULL_ADDR_NOUNIT,COMBINED_ADDR_NOUNIT
FEATURE_KE,,,,,,,,,,,,,,,,,,,,,
2866,2866,0,3026,1,ACTIVE,NaN,51,NaN,NaN,NaN,...,DRE,DUP,51 BOUNDARY ST,NaN,0.0,0.0,POINT (1355944.2850821316 410373.4892156422),51 BOUNDARY ST CITY OF PITTSBURGH 15213,51 BOUNDARY ST,51 BOUNDARY ST CITY OF PITTSBURGH 15213


In [836]:
address_points['COMBINED_ADDR'].head()

FEATURE_KE
2866        51 BOUNDARY ST CITY OF PITTSBURGH 15213
4022     2201 SALISBURY ST CITY OF PITTSBURGH 15210
5664    1700 WASHINGTON BLVD PORT VUE BOROUGH 15133
5688         1309 FREEPORT RD HARMAR TOWNSHIP 15024
6902          298 HARMONY RD KILBUCK TOWNSHIP 15237
Name: COMBINED_ADDR, dtype: object

In [837]:
len(address_points)

645

In [ ]:
# WARNING: will call API; control request size
# results = address_points['COMBINED_ADDR'].apply(geolocator.geocode, components=geo_components)

In [806]:
results_x = results.apply(lambda loc: loc.longitude)
results_y = results.apply(lambda loc: loc.latitude)

In [838]:
address_points.loc[:, 'geometry'] = geopandas.points_from_xy(results_x, results_y)

In [ ]:
address_points = address_points.set_geometry('geometry', crs="EPSG:4326")

In [844]:
address_points.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
address_points.to_file('data/processing/AddressPoints_dupes_geocoded.geojson')

### Property Addresses

In [ ]:
properties = pd.read_csv('data/processing/prop_subset_toGeocode.csv', index_col=0)
properties.head(1)

,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYUNIT,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYZIP,MUNICODE,MUNIDESC,NEIGHCODE,...,HALFBATHS,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TAXYEAR,GEO_MERGE_KEY,PARTIAL_ADDRESS
PARID,,,,,,,,,,,,,,,,,,,,,
0002H00033000000,1516,NaN,NaN,5TH AVE,PITTSBURGH,PA,15219,101,1st Ward - PITTSBURGH,51C04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,1516 5TH AVE 15219,1516 5TH AVE 15219


In [10]:
properties['PARTIAL_ADDRESS'].head()

PARID
0002H00033000000         1516 5TH AVE 15219
0002H00034000000         1520 5TH AVE 15219
0002L00316000000        1234 SEITZ ST 15219
0009N00148000000         923 PENN AVE 15222
0002A00097000000    527 SMITHFIELD ST 15222
Name: PARTIAL_ADDRESS, dtype: object

In [11]:
len(properties)

661

In [ ]:
# WARNING: will call API; control request size
# prop_results = properties['PARTIAL_ADDRESS'].apply(geolocator.geocode, components=geo_components)

In [32]:
prop_results

PARID
0002H00033000000    (1516 Fifth Ave, Pittsburgh, PA 15219, USA, (4...
0002H00034000000    (1520 Fifth Ave, Pittsburgh, PA 15219, USA, (4...
0002L00316000000    (1234 Seitz St, Pittsburgh, PA 15219, USA, (40...
0009N00148000000    (923 Penn Ave, Pittsburgh, PA 15222, USA, (40....
0002A00097000000    (Oliver Building, 527 Smithfield St, Pittsburg...
                                          ...                        
1100C00296000000    (Oak Rd, Plum, PA 15239, USA, (40.480955, -79....
0571N00040000000    (1049 Boyce Rd, Pittsburgh, PA 15241, USA, (40...
0776B00010000000    (2760 Washington Rd, Pittsburgh, PA 15241, USA...
0193C00010000000    (900 Cedar Blvd, Pittsburgh, PA 15228, USA, (4...
1999B00010000000    (130 Mt Pleasant Rd, Warrendale, PA 15086, USA...
Name: PARTIAL_ADDRESS, Length: 661, dtype: object

In [100]:
# cache results
with shelve.open('cache/geocode') as cache:
    cache['prop_results'] = prop_results

In [18]:
prop_x = prop_results.apply(lambda loc: loc.longitude)
prop_y = prop_results.apply(lambda loc: loc.latitude)

In [19]:
properties.loc[:, 'geometry'] = geopandas.points_from_xy(prop_x, prop_y)

In [23]:
properties = properties.set_geometry('geometry', crs="EPSG:4326")

In [ ]:
properties.to_file('data/processing/prop_subset_geocoded.geojson')

## Owner Addresses

In [36]:
owners = geopandas.GeoDataFrame(pd.read_csv('data/processing/prop_owners_top.csv', index_col=0))

In [ ]:
owners['OWNER_CLEANADDR'].head()

OWNER_ID
629560013250            414 GRANT ST RM 200 PITTSBURGH PA
377568607449                    PO BOX 4900 SCOTTSDALE AZ
269480886837    412 BLVD OF THE ALLIES FL 7 PITTSBURGH PA
250684894979                829 INDUSTRY ST PITTSBURGH PA
335109363737                 901 BINGHAM ST PITTSBURGH PA
Name: OWNER_CLEANADDR, dtype: object

In [340]:
async def geocode(client, address):
    async with SEM:
        url = "https://maps.googleapis.com/maps/api/geocode/json"
        params = {
            'address': address, 
            'components': geo_components, 
            'key': API_KEY
        }
        
        try:
            r = await client.get(url, params=params)
            r.raise_for_status()
            data = r.json()
            # data = {
            #     'place_id': 'id123',
            #     'geometry': {
            #         'location': {'lat': 0, 'lng': 0},
            #     },
            #     'norm_address': address
            # }
            
        except httpx.RequestError:
            # network issue (timeout, DNS, etc.)
            return {'error': 'request_error', 'input': address}

        except httpx.HTTPStatusError:
            # non-200 response
            return {'error': 'http_error', 'input': address}

        # API-level status check
        status = data.get('status')
        
        if status == 'ZERO_RESULTS':
            return {'error': 'ZERO_RESULTS', 'input': address}

        if status != 'OK':
            return {'error': status, 'input': address}

        result = data['results'][0]
        return {
            'formatted_address': result['formatted_address'],
            'place_id': result.get('place_id'),
            'latitude': result['geometry']['location']['lat'],
            'longitude': result['geometry']['location']['lng']
        }

In [ ]:
async def get_place_details(client, place_id):
    url = f"https://places.googleapis.com/v1/places/{place_id}"

    headers = {
        'X-Goog-Api-Key': API_KEY,
        'X-Goog-FieldMask': 'displayName,types,formattedAddress,addressComponents,containingPlaces'
    }

    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()

In [341]:
async def batch_geocode(addresses):
    async with httpx.AsyncClient(timeout=10) as client:
        tasks = [geocode(client, addr) for addr in addresses]
        results = await asyncio.gather(*tasks)
        results_df = pd.DataFrame(results)
        results_df.index = addresses.index
        
        return results_df

In [ ]:
# WARNING: will call API; control request size
# owner_geos = await batch_geocode(owners['OWNER_CLEANADDR'])

In [353]:
# cache results
with shelve.open('cache/geocode') as cache:
    cache['owner_geos'] = owner_geos

In [354]:
owners.loc[:, 'geometry'] = geopandas.points_from_xy(owner_geos['longitude'], owner_geos['latitude'])

In [360]:
owners = owners.set_geometry('geometry', crs="EPSG:4326")

In [362]:
owners.head()

,OWNER_CLEANADDR,OWNER_STREETADDR,OWNER_PLACE,OWNER_ZIP,OWNER_STATE,OWNER_CITY,geometry
OWNER_ID,,,,,,,
629560013250,414 GRANT ST RM 200 PITTSBURGH PA,414 GRANT ST RM 200,PITTSBURGH PA,NaN,PA,PITTSBURGH,POINT (-79.99717 40.43823)
377568607449,PO BOX 4900 SCOTTSDALE AZ,PO BOX 4900,SCOTTSDALE AZ,NaN,AZ,SCOTTSDALE,POINT (-111.92167 33.49488)
269480886837,412 BLVD OF THE ALLIES FL 7 PITTSBURGH PA,412 BLVD OF THE ALLIES FL 7,PITTSBURGH PA,NaN,PA,PITTSBURGH,POINT (-79.99964 40.43741)
250684894979,829 INDUSTRY ST PITTSBURGH PA,829 INDUSTRY ST,PITTSBURGH PA,NaN,PA,PITTSBURGH,POINT (-79.99272 40.42093)
335109363737,901 BINGHAM ST PITTSBURGH PA,901 BINGHAM ST,PITTSBURGH PA,NaN,PA,PITTSBURGH,POINT (-79.98965 40.42956)


In [363]:
owners.to_file('data/processing/prop_owners_top_geocoded.geojson')

In [ ]:
# async def process_addresses(addresses):
#     async with httpx.AsyncClient(timeout=10) as client:
#         idx = addresses.index
#         vals = addresses.values
        
#         # geocode
#         tasks = []
#         for addr in vals:
#             tasks.append(geocode(client, addr))

#         geos = await asyncio.gather(*tasks)

#         # get place details from place id
#         tasks = []
#         for geo in geos:
#             if geo and geo['place_id']:
#                 tasks.append(get_place_details(client, geo['place_id']))
#             else:
#                 tasks.append(asyncio.sleep(0, result=None))

#         places = await asyncio.gather(*tasks)

#         return list(zip(idx, geos, places))

In [ ]:
# WARNING: will call API; control request size
# owner_results = await process_addresses(owners['OWNER_CLEANADDR'].head(3))

In [ ]:
# cache results
# with shelve.open('cache/geocode') as cache:
#     cache['owner_results'] = owner_results

In [ ]:
# with shelve.open('cache/geocode_owners') as cache:
#     for idx, addr in owners['OWNER_CLEANADDR'].head().items():
#         if idx in cache:
#             continue

#         result = process_addresses(addr)
#         cache[idx] = result